In [ ]:
!pip install -U langgraph langchain-google-genai langchain-community ddgs

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    question: str
    language: str
    answer: str

In [ ]:
def router(state):
    q = state["question"].lower()

    if "python" in q:
        return {"language": "PYTHON"}
    elif "java" in q:
        return {"language": "JAVA"}
    else:
        return {"language": "OTHER"}

In [ ]:
def python_tutor(state):
    response = model.invoke(
        f"Answer this Python question in one simple sentence: {state['question']}"
    )
    return {"answer": response.content}


def java_tutor(state):
    response = model.invoke(
        f"Answer this Java question in one simple sentence: {state['question']}"
    )
    return {"answer": response.content}

In [ ]:
def other(state):
    return {"answer": "Please ask a Python or Java programming question."}

In [ ]:
def route(state):
    if state["language"] == "PYTHON":
        return "python"
    elif state["language"] == "JAVA":
        return "java"
    else:
        return "other"

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

graph.add_node("router", router)
graph.add_node("python", python_tutor)
graph.add_node("java", java_tutor)
graph.add_node("other", other)

graph.add_edge(START, "router")

graph.add_conditional_edges(
    "router",
    route,
    {
        "python": "python",
        "java": "java",
        "other": "other"
    }
)

graph.add_edge("python", END)
graph.add_edge("java", END)
graph.add_edge("other", END)

app = graph.compile()

In [ ]:
result = app.invoke({
    "question": "Python: Explain variables"
})

print(result["answer"])

[{'type': 'text', 'text': 'In Python, a variable is a named container used to store and reference data values.', 'extras': {'signature': 'EtINCs8NAWkUfRPzapuRLzMrliI9BtWnoBC4K6REn5SagmOAqwLmVOkb5Uhe9gnxe/pe1Ch/bSaTBm+/p6Q56sqE5Be1y7CoPZDX2/oiBL26qJJDrVuUkThY7FqWgCZPQBYTKdGrUSWC9hWTq+A+DiAEGqqI/074ZSaESZuFeBHGAFs3HrFDOAequvK+fPg4+cujFP0IznPMbWFMnBVks/IPNi/W3EBSFvPh2l4nbl0ftrfo311ZYhwbKAwtwrwPwpdgL5S6fVsdRwn7SwitLAEhNPaQweEPm3iPOsQ6ecNiW0cmFf1UMpqfl/m1skonsw7He2nLvLuSt2jDm4co4sjfZR/aU3q35R/LAab48rxf9kr/KCD1kQz7j9uGvW7U1/wYqUBSQZBufexqXUfz5Ta7KIZ7CYFeJdKUiLszhgBQgigkQzaHohv/X4K3z2BtYpwdwyWys6Fwv2td+rpuoTK+EAeFrwdTdXLN1Ifza72PXR3g26H1KWdgVN+eH4VAQEBbpxH9rvM2CC5xiHG7rAQWfaR3l8ErnnZAbfmQ8OJeHI/X2HUOUEE2HYSe64Rz7W1peuIBaBO4JxiztqnG0DzoXrXwcbES+DEilQcasASjMgJKO4EMTDz0JjPXT1iGlFOe1OtVWN78v+oTB0ITRZ4OLp6U34psbNgEyFtqI+1+uCtN4vnhrm2dVik4sn7OfEao8LDZTEbpsy9UNdyPO90oA0u4DCnkzdx76k5Gk6ScA6E9S4i0XPq4Kcs/3OjzdmCP73C06O82iVnIhN4HqrYavFdc6MwK9zPcqEw83HQWvNsrNyvNf6EJWFL4lA0cxzhzTqQAvQBhemystwyEFkACJ7AKW5